In [23]:
import os
import pandas as pd
from utils import take_until_punct_or_space

def is_matched_str(pred_tokens, gold_tokens, birdirectional=True):
    if any(" ".join(gold_tokens) == " ".join(pred_tokens[i:i+len(gold_tokens)]) for i in range(len(pred_tokens))):
        return True
    elif birdirectional and any(" ".join(pred_tokens) == " ".join(gold_tokens[i:i+len(pred_tokens)]) for i in range(len(gold_tokens))):
        return True
    return False

def partial_match(pred, golds, birdirectional=True):
    return any(is_matched_str(pred, gold, birdirectional) for gold in golds)

def partial_match_scores(predictions, gold_answers, birdirect=False):
    scores = []
    for prediction, _gold_answers in zip(predictions, gold_answers):
        try:
            prediction = prediction.tolist()
        except Exception:
            assert isinstance(prediction, str)
            prediction = [prediction]

        if len(prediction) == 0:
            scores.append(0)
            continue

        score = partial_match(prediction, _gold_answers, birdirect)        
        scores.append(int(score))
    return sum(scores)/len(scores)

def partial_match_scores_use_generation(predictions, gold_answers, birdirect=False):
    scores = []
    for generations, _gold_answers in zip(predictions, gold_answers):
        generations = take_until_punct_or_space(generations[0])
        if len(generations) == 0:
            scores.append(0)
            continue
        score = partial_match(generations, _gold_answers, birdirect)
        scores.append(int(score))
    
    return sum(scores)/len(scores)


In [20]:
fileroot = "/home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama/llama3.2_1b"
filename = os.path.join(fileroot, "myriadlama.singleparaqapair.1fshots.5samples.5paras.feather")
df = pd.read_feather(filename)

In [21]:
df

,uuid,answers,prediction,generation,templates,prompt,paraphrases,predict_lemma,answer_lemmas,generation_lemmas
0,5d09424876ba7f4-15e8746c51e1d4c,"[NBC, National Broadcasting Company]",NBC,NBC,[[This Is Your Life premiered on the network [...,"Based on the context, predict the [MASK] in th...",[This Is Your Life premiered on the network [M...,[nbc],"[[nbc], [national, broadcasting, company]]",[nbc]
1,5d09424876ba7f4-15e8746c51e1d4c,"[NBC, National Broadcasting Company]",This,This Is Your Life,[[Which service originally aired This Is Your ...,"Based on the context, predict the [MASK] in th...",[Which service originally aired This Is Your L...,[this],"[[nbc], [national, broadcasting, company]]","[this, be, your, life]"
2,5d09424876ba7f4-15e8746c51e1d4c,"[NBC, National Broadcasting Company]",[MASK],[MASK],[[[MASK] had the distinction of being the firs...,"Based on the context, predict the [MASK] in th...",[[MASK] had the distinction of being the first...,"[[, mask, ]]","[[nbc], [national, broadcasting, company]]","[[, mask, ]]"
3,5d09424876ba7f4-15e8746c51e1d4c,"[NBC, National Broadcasting Company]",NBC,NBC,[[The first episode of This Is Your Life aired...,"Based on the context, predict the [MASK] in th...",[The first episode of This Is Your Life aired ...,[nbc],"[[nbc], [national, broadcasting, company]]",[nbc]
4,5d09424876ba7f4-15e8746c51e1d4c,"[NBC, National Broadcasting Company]",Netflix,Netflix,[[[MASK] had the distinction of being the firs...,"Based on the context, predict the [MASK] in th...",[[MASK] had the distinction of being the first...,[netflix],"[[nbc], [national, broadcasting, company]]",[netflix]
...,...,...,...,...,...,...,...,...,...,...
9995,9e74549729da44f-cdcb59adf18e302,"[Russian, Russian language, ru, Russkiy, Russk...",Russian,Russian,[[Uspekhi Fizicheskikh Nauk is created in lang...,"Based on the context, predict the [MASK] in th...",[Uspekhi Fizicheskikh Nauk is created in langu...,[russian],"[[russian], [russian, language], [ru], [russki...",[russian]
9996,9e74549729da44f-cdcb59adf18e302,"[Russian, Russian language, ru, Russkiy, Russk...",Russian,Russian,[[The original language of Uspekhi Fizicheskik...,"Based on the context, predict the [MASK] in th...",[The original language of Uspekhi Fizicheskikh...,[russian],"[[russian], [russian, language], [ru], [russki...",[russian]
9997,9e74549729da44f-cdcb59adf18e302,"[Russian, Russian language, ru, Russkiy, Russk...",Russian,Russian,[[The original language of Uspekhi Fizicheskik...,"Based on the context, predict the [MASK] in th...",[The original language of Uspekhi Fizicheskikh...,[russian],"[[russian], [russian, language], [ru], [russki...",[russian]
9998,9e74549729da44f-cdcb59adf18e302,"[Russian, Russian language, ru, Russkiy, Russk...",[MASK],[MASK] language,[[Uspekhi Fizicheskikh Nauk is an artifact of ...,"Based on the context, predict the [MASK] in th...",[Uspekhi Fizicheskikh Nauk is an artifact of t...,"[[, mask, ]]","[[russian], [russian, language], [ru], [russki...","[[, mask, ], language]"


In [24]:
answers = [[answer.tolist() for answer in answers.tolist()] for answers in df["answer_lemmas"]]

# predictions = df["predict_lemma"].tolist()
# acc = partial_match_scores(predictions, answers, birdirect=True)
# print("Using prediction lemmas:", acc)

generations = [[pred.tolist()] for pred in df["generation_lemmas"].tolist()]
acc = partial_match_scores_use_generation(generations, answers, birdirect=True)
print("Using generation lemmas:", acc)

Using generation lemmas: 0.1391
